# Chapter 7 &mdash; Brzozowski's Minimization: Reverse, Determinize, Twice

**Concept 10 of the Chapter 7 decomposition:** *Brzozowski's Minimization: Reverse, Determinize, Reverse, Determinize*

`nfa2dfa(rev_dfa(nfa2dfa(rev_dfa(D))))` &mdash; minimization with no distinguishability table at all.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Brzozowski-Minimization/Concept-Brzozowski-Minimization.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


**Brzozowski's algorithm** is startling: to minimize a DFA, **reverse it, determinize,
reverse again, determinize again.** That is the whole algorithm.

$$\text{min}(D) = \text{det}(\text{rev}(\text{det}(\text{rev}(D))))$$

No frames, no distinguishability table, no equivalence classes. The reason it works:
determinizing a **reversed** machine automatically produces a machine with no two
equivalent states, because subset-construction states are distinguished by the
*suffixes* they accept.

Jove packages it as `min_dfa_brz`.

## 2. Definitions

### A deliberately bloated DFA

In [ ]:
blimp = md2mc('''DFA
I  : 0 -> A
I  : 1 -> B
A  : 0 -> C
A  : 1 -> D
B  : 0 -> D
B  : 1 -> C
C  : 0 | 1 -> F1
D  : 0 | 1 -> F2
F1 : 0 | 1 -> F1
F2 : 0 | 1 -> F2
''')
print("|Q| =", len(blimp["Q"]))

### The four steps, spelled out

In [ ]:
def brz(D):
    s1 = rev_dfa(D)          # NFA
    s2 = nfa2dfa(s1)         # DFA
    s3 = rev_dfa(s2)         # NFA
    s4 = nfa2dfa(s3)         # DFA -- and it is minimal
    return s1, s2, s3, s4

## 3. Tests

Watch the size at each step.

In [ ]:
s1, s2, s3, s4 = brz(blimp)
print("original          : %2d states" % len(blimp["Q"]))
print("1. rev  (NFA)     : %2d states" % len(s1["Q"]))
print("2. det            : %2d states" % len(s2["Q"]))
print("3. rev  (NFA)     : %2d states" % len(s3["Q"]))
print("4. det  (minimal) : %2d states" % len(s4["Q"]))

The result really is minimal &mdash; same size as `min_dfa`.

In [ ]:
m = min_dfa(blimp)
print("min_dfa      : %d states" % len(m["Q"]))
print("Brzozowski   : %d states" % len(s4["Q"]))
assert len(s4["Q"]) == len(m["Q"])
assert iso_dfa(s4, m)
print("isomorphic to min_dfa's answer? ", iso_dfa(s4, m))

And the language is untouched.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
assert all(accepts_dfa(s4, s) == accepts_dfa(blimp, s) for s in strs)
print("same language on all %d strings up to length 10" % len(strs))

Jove's `min_dfa_brz` is the same four steps in one call.

In [ ]:
b = min_dfa_brz(blimp)
print("min_dfa_brz : %d states, isomorphic to min_dfa: %s"
      % (len(b["Q"]), iso_dfa(b, m)))
assert iso_dfa(b, m)

Doing only **one** reverse-determinize is not enough.

In [ ]:
half = nfa2dfa(rev_dfa(blimp))
print("after one round : %d states (minimal would be %d)" % (len(half["Q"]), len(m["Q"])))
print("the language is the REVERSE at that point, so it cannot be the answer.")

## 4. Animation

The minimal machine Brzozowski's algorithm produces.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa_brz(blimp), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Run the four steps on a machine that is already minimal. What happens?
2. Why does determinizing a reversed DFA remove equivalent states automatically?
3. Compare the cost of Brzozowski with the frame algorithm on a 10-state DFA.

In [ ]:
# Your work for the exercises above.